<a href="https://colab.research.google.com/github/ramgeethatamatam-rgb/-used-cars-data-prep/blob/main/Datacleaning_and_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1 — Load, Inspect, and Clean**


In [51]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

import kagglehub

# Download latest version
path = kagglehub.dataset_download("manishkr1754/cardekho-used-car-data")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'cardekho-used-car-data' dataset.
Path to dataset files: /kaggle/input/cardekho-used-car-data


In [52]:
df = pd.read_csv(os.path.join(path, 'cardekho_dataset.csv'))

In [53]:
print(df.shape)

(15411, 14)


In [54]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB
None


In [55]:
df.isnull().sum()

,0
Unnamed: 0,0
car_name,0
brand,0
model,0
vehicle_age,0
km_driven,0
seller_type,0
fuel_type,0
transmission_type,0
mileage,0


In [56]:
df.isnull().sum() * 100 / len(df)

,0
Unnamed: 0,0.0
car_name,0.0
brand,0.0
model,0.0
vehicle_age,0.0
km_driven,0.0
seller_type,0.0
fuel_type,0.0
transmission_type,0.0
mileage,0.0


In [57]:
df.describe()

,Unnamed: 0,vehicle_age,km_driven,mileage,engine,max_power,seats,selling_price
count,15411.000000,15411.000000,1.541100e+04,15411.000000,15411.000000,15411.000000,15411.000000,1.541100e+04
mean,9811.857699,6.036338,5.561648e+04,19.701151,1486.057751,100.588254,5.325482,7.749711e+05
std,5643.418542,3.013291,5.161855e+04,4.171265,521.106696,42.972979,0.807628,8.941284e+05
min,0.000000,0.000000,1.000000e+02,4.000000,793.000000,38.400000,0.000000,4.000000e+04
25%,4906.500000,4.000000,3.000000e+04,17.000000,1197.000000,74.000000,5.000000,3.850000e+05
50%,9872.000000,6.000000,5.000000e+04,19.670000,1248.000000,88.500000,5.000000,5.560000e+05
75%,14668.500000,8.000000,7.000000e+04,22.700000,1582.000000,117.300000,5.000000,8.250000e+05
max,19543.000000,29.000000,3.800000e+06,33.540000,6592.000000,626.000000,9.000000,3.950000e+07


In [58]:
df = df.dropna(subset = ['selling_price'])

For columns mileage, engine, and max_power — strip any unit strings (e.g. "kmpl", "CC", "bhp"), convert to numeric using pd.to_numeric(..., errors='coerce'), and fill remaining nulls with each column's median.

In [59]:
import pandas as pd

cols = ['mileage', 'engine', 'max_power']

for col in cols:
    df[col] = df[col].astype(str)
    df[col] =df[col].str.extract(r'(\d+\.\d+)')
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())


In [60]:
df[['mileage', 'engine', 'max_power']].head()

,mileage,engine,max_power
0,19.70,NaN,46.30
1,18.90,NaN,82.00
2,17.00,NaN,80.00
3,20.92,NaN,67.10
4,22.77,NaN,98.59


In [61]:
df['engine'].unique()[:10]

array([nan])

In [62]:
for col in ['engine', 'mileage', 'max_power']:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Filled {col} missing values with median: {median_val}")

Filled engine missing values with median: nan
Filled mileage missing values with median: 19.67
Filled max_power missing values with median: 88.5


Remove rows where selling_price is 999999999 or below 10000.

In [63]:
df = df[(df['selling_price'] != 999999999) & (df['selling_price'] >= 10000)]

print(df['selling_price'].min())
print(df['selling_price'].max())

40000
39500000


In [64]:
df = df.drop_duplicates()

In [65]:
df.duplicated().sum()

np.int64(0)

In [66]:
print(df.shape)

(15411, 14)


Task 2 —** Encode Categorical Features**

Apply label encoding to transmission_type — map Manual → 0 and Automatic → 1.

In [67]:
print("Before Encoding")
print(df['transmission_type'].value_counts())

df['transmission_type'] = df ['transmission_type'].map({'Manual': 0, 'Automatic':1})

print("After Encoding")
print(df['transmission_type'].value_counts())


Before Encoding
transmission_type
Manual       12225
Automatic     3186
Name: count, dtype: int64
After Encoding
transmission_type
0    12225
1     3186
Name: count, dtype: int64


Apply one-hot encoding to fuel_type and seller_type using pd.get_dummies() with drop_first=True.

In [68]:
df = pd.get_dummies(df, columns=['fuel_type', 'seller_type'], drop_first=True)

In [69]:
df.head(10)

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,transmission_type,mileage,engine,max_power,seats,selling_price,fuel_type_Diesel,fuel_type_Electric,fuel_type_LPG,fuel_type_Petrol,seller_type_Individual,seller_type_Trustmark Dealer
0,0,Maruti Alto,Maruti,Alto,9,120000,0,19.70,NaN,46.30,5,120000,False,False,False,True,True,False
1,1,Hyundai Grand,Hyundai,Grand,5,20000,0,18.90,NaN,82.00,5,550000,False,False,False,True,True,False
2,2,Hyundai i20,Hyundai,i20,11,60000,0,17.00,NaN,80.00,5,215000,False,False,False,True,True,False
3,3,Maruti Alto,Maruti,Alto,9,37000,0,20.92,NaN,67.10,5,226000,False,False,False,True,True,False
4,4,Ford Ecosport,Ford,Ecosport,6,30000,0,22.77,NaN,98.59,5,570000,True,False,False,False,False,False
5,5,Maruti Wagon R,Maruti,Wagon R,8,35000,0,18.90,NaN,67.10,5,350000,False,False,False,True,True,False
6,6,Hyundai i10,Hyundai,i10,8,40000,0,20.36,NaN,78.90,5,315000,False,False,False,True,False,False
7,7,Maruti Wagon R,Maruti,Wagon R,3,17512,0,20.51,NaN,67.04,5,410000,False,False,False,True,False,False
8,8,Hyundai Venue,Hyundai,Venue,2,20000,1,18.15,NaN,118.35,5,1050000,False,False,False,True,True,False
9,12,Maruti Swift,Maruti,Swift,4,28321,0,16.60,NaN,85.00,5,511000,False,False,False,True,False,False


Print the column list of the final DataFrame using df.columns.tolist().

In [70]:
print(df.columns.tolist())

['Unnamed: 0', 'car_name', 'brand', 'model', 'vehicle_age', 'km_driven', 'transmission_type', 'mileage', 'engine', 'max_power', 'seats', 'selling_price', 'fuel_type_Diesel', 'fuel_type_Electric', 'fuel_type_LPG', 'fuel_type_Petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer']


In a markdown cell, answer: Why is drop_first=True used in one-hot encoding, and what does a row of all zeros in the new dummy columns represent?


ANS:Removes categorical variables to avoid the multiple dummy variables(0/1) columns including all categories can create redundency.This is called multicollinearity.

A row with all zeros in a new dummy columns represent the data belongs to the dropped category

**Task 3 — Split and Compute Baseline MAE**

In [71]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import numpy as np

# 1. Define X and y
X = df.drop('selling_price', axis =1)
y = df['selling_price']

# 2. Drop non-numeric columns from X
X = X.select_dtypes(include=[np.number])

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. Baseline model (predict mean of y_train)
baseline_pred = np.full(len(y_test), y_train.mean())

# 5. Calculate MAE
mae = mean_absolute_error(y_test, baseline_pred)

# 6. Print result
print(f"Baseline MAE: ₹{round(mae)}")

Baseline MAE: ₹468748
